# Environmental Equity Economy 그림과 표
이 노트북과 같은 폴더의 data 스냅샷을 읽습니다. GAMS를 다시 풀지 않습니다.
패키지: pandas, numpy, reportlab. 설치: python -m pip install pandas numpy reportlab
C=(r=90%, REN=60%), D=(r=90%, REN=100%). HH90=(r=90%, REN=0%).
Overall outcomes의 H는 r=100% HH 단독이므로 HH90과 구분합니다.
실행 경로를 바꿨다면 아래 BASE만 수정하세요. 그림은 PDF/SVG와 가능한 경우 300dpi PNG로 저장합니다.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from reportlab.graphics.shapes import Drawing, String, Rect, Line, Polygon, Group
from reportlab.graphics.charts.lineplots import LinePlot
from reportlab.graphics.charts.barcharts import VerticalBarChart
from reportlab.graphics.widgets.markers import makeMarker
from reportlab.graphics import renderPDF,renderSVG
from reportlab.lib import colors
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'Run_Master.gms').is_file())
BASE = PROJECT/'Figure'/'Sections_2_4'
OUT=BASE/'exports'
PLOT_DATA=BASE/'figure_data'
OUT.mkdir(parents=True,exist_ok=True)
PLOT_DATA.mkdir(parents=True,exist_ok=True)
s=pd.read_csv(BASE/'data/policy_summary.csv')
gen=pd.read_csv(BASE/'data/generation_korea.csv')
low=pd.read_csv(BASE/'data/low_income_summary.csv')
joint=pd.read_csv(BASE/'data/joint_summary.csv')
quintile=pd.read_csv(BASE/'data/quintile_summary.csv')
info=json.loads((BASE/'data/cge_definitions.json').read_text(encoding='utf-8'))
micro_info=json.loads((BASE/'data/micro_definitions.json').read_text(encoding='utf-8'))
factor=info['model_money_to_billion_usd_conditional']
bau=s.loc[s.policy_id.eq('BAU')].iloc[0]
tou=s.loc[s.policy_id.eq('TOU')].iloc[0]
s['CO2_reduction_Mt']=-s.CO2_change_Mt
s['power_CO2_reduction_Mt']=-s.power_CO2_change_Mt
s['other_CO2_reduction_Mt']=s.CO2_reduction_Mt-s.power_CO2_reduction_Mt
s['SG_change_billion_USD']=(s.SG-bau.SG)*factor
s['loss_relief_pct']=(s.EV-tou.EV)/(-tou.EV)*100
POLICIES=['TOU','r90_a00','r90_a60','r90_a100']
NAMES={'TOU':'TOU','r90_a00':'HH90','r90_a60':'C','r90_a100':'D'}
ALLOC=['U','I','IC']
PALETTE=[colors.HexColor(x) for x in ['#82909B','#9DBAD1','#39769D','#173B63']]
ACOL={'U':colors.HexColor('#8B9299'),'I':colors.HexColor('#2A79A8'),'IC':colors.HexColor('#A85233')}
NAVY=colors.HexColor('#173B63');GREY=colors.HexColor('#53616D');GRID=colors.HexColor('#E1E6EB')
FIGURES={}

def txt(*args,**kwargs):
    kwargs.setdefault('fontName','Helvetica');return String(*args,**kwargs)
def canvas(w=680,h=330):
    d=Drawing(w,h);d.add(Rect(0,0,w,h,fillColor=colors.white,strokeColor=None));return d
def vertical(d,x,y,text,size=10):
    d.add(Group(txt(0,0,text,textAnchor='middle',fontSize=size),transform=(0,1,-1,0,x,y)))
def lineplot(d,x,y,w,h,data,xrange,yrange,xticks,yticks,line_colors,dashes=None):
    p=LinePlot();p.x=x;p.y=y;p.width=w;p.height=h;p.data=data
    p.xValueAxis.valueMin,p.xValueAxis.valueMax=xrange
    p.yValueAxis.valueMin,p.yValueAxis.valueMax=yrange
    p.xValueAxis.valueSteps=xticks;p.yValueAxis.valueSteps=yticks
    p.xValueAxis.labelTextFormat='%g';p.yValueAxis.labelTextFormat='%g'
    for axis in [p.xValueAxis,p.yValueAxis]:
        axis.labels.fontName='Helvetica';axis.labels.fontSize=8.5;axis.strokeColor=GREY;axis.strokeWidth=.6
    p.yValueAxis.visibleGrid=True;p.yValueAxis.gridStrokeColor=GRID;p.yValueAxis.gridStrokeWidth=.4
    for i,col in enumerate(line_colors):
        p.lines[i].strokeColor=col;p.lines[i].strokeWidth=1.25
        p.lines[i].symbol=makeMarker('FilledCircle');p.lines[i].symbol.size=4.2
        p.lines[i].symbol.fillColor=col;p.lines[i].symbol.strokeColor=col
        if dashes and dashes[i]:p.lines[i].strokeDashArray=dashes[i]
    d.add(p)
    return p

def bars(d,x,y,w,h,data,categories,line_colors,yrange,ticks):
    p=VerticalBarChart();p.x=x;p.y=y;p.width=w;p.height=h;p.data=data
    p.valueAxis.valueMin,p.valueAxis.valueMax=yrange;p.valueAxis.valueSteps=ticks
    p.valueAxis.labelTextFormat='%g';p.valueAxis.labels.fontName='Helvetica';p.valueAxis.labels.fontSize=8
    p.valueAxis.strokeColor=GREY;p.valueAxis.visibleGrid=True;p.valueAxis.gridStrokeColor=GRID;p.valueAxis.gridStrokeWidth=.4
    p.categoryAxis.categoryNames=categories;p.categoryAxis.visibleLabels=False;p.categoryAxis.visibleTicks=False
    p.categoryAxis.strokeColor=None;p.groupSpacing=9;p.barSpacing=1
    for i,col in enumerate(line_colors):p.bars[i].fillColor=col;p.bars[i].strokeColor=None
    d.add(p)
    for i,name in enumerate(categories):d.add(txt(x+(i+.5)*w/len(categories),y-15,name,textAnchor='middle',fontSize=8.5))
    yy=y+(0-yrange[0])/(yrange[1]-yrange[0])*h
    d.add(Line(x,yy,x+w,yy,strokeColor=GREY,strokeWidth=.5))
    return p

def legend(d,labels,line_colors,y=302,x=70,gap=140):
    for i,(label,col) in enumerate(zip(labels,line_colors)):
        xx=x+i*gap;d.add(Line(xx,y,xx+17,y,strokeColor=col,strokeWidth=1.5))
        d.add(txt(xx+23,y-3,label,fontSize=9,fillColor=GREY))

def export_figure(name,d):
    FIGURES[name]=d
    renderPDF.drawToFile(d,str(OUT/(name+'.pdf')))
    renderSVG.drawToFile(d,str(OUT/(name+'.svg')))

## 2 Environmental outcomes
Figure 5.3: 전체 격자의 한국 CO2 감축. r=0은 같은 TOU의 반복 표시입니다.
Figure 5.4: 정부저축 보전 비교에서 TOU, HH90, C, D의 발전원별 생산 변화.
발전은 BAU 대비 수량 변화율입니다. REN은 재생 및 기타 발전 집계이며 원자력은 지원하지 않습니다.

In [ ]:
policy=s.loc[s.rate.ne('base')].copy()
heat=policy.pivot(index='recycling_fraction',columns='ren_fraction',values='CO2_reduction_Mt').sort_index()
heat.to_csv(PLOT_DATA/'Figure_5_3_CO2_heatmap.csv',encoding='utf-8-sig')
heat_long=policy[['policy_id','recycling_fraction','ren_fraction','CO2_reduction_Mt','SG']].copy()
heat_long['SG_preserved']=heat_long.SG.ge(bau.SG-1e-10)
heat_long.to_csv(PLOT_DATA/'Figure_5_3_points.csv',index=False,encoding='utf-8-sig')
d=canvas(680,420)
d.add(txt(70,397,'CO2 reductions across the recycling and allocation grid',fontName='Helvetica-Bold',fontSize=13,fillColor=NAVY))
x0,y0,cw,ch=87,61,37,26
for i,r in enumerate(heat.index):
    for j,a in enumerate(heat.columns):
        value=heat.loc[r,a];t=np.clip(value/13,0,1)
        rgb=(np.array([237,247,241])+(np.array([11,98,66])-np.array([237,247,241]))*t)/255
        d.add(Rect(x0+j*cw,y0+i*ch,cw,ch,fillColor=colors.Color(*rgb),strokeColor=colors.white,strokeWidth=.6))
for i,v in enumerate(range(0,101,10)):
    d.add(txt(x0+(i+.5)*cw,y0-15,str(v),textAnchor='middle',fontSize=9))
    d.add(txt(x0-9,y0+(i+.5)*ch-3,str(v),textAnchor='end',fontSize=9))
d.add(txt(x0+5.5*cw,20,'REN allocation share a (%)',textAnchor='middle',fontSize=11))
vertical(d,22,y0+5.5*ch,'Recycling share r (%)',11)
by=y0+10*ch
d.add(Line(x0,by,x0+11*cw,by,strokeColor=colors.black,strokeWidth=1,strokeDashArray=[4,2]))
d.add(txt(92,363,'Top row: SG below BAU; remaining sampled rows preserve SG',fontSize=9,fillColor=GREY))
for label,r,a in [('A',1,.5),('B',1,1),('C',.9,.6),('D',.9,1)]:
    xx=x0+(round(a*10)+.5)*cw;yy=y0+(round(r*10)+.5)*ch
    d.add(txt(xx,yy-4,label,textAnchor='middle',fontName='Helvetica-Bold',fontSize=11,fillColor=colors.white))
for i in range(100):
    t=i/99;rgb=(np.array([237,247,241])+(np.array([11,98,66])-np.array([237,247,241]))*t)/255
    d.add(Rect(542,y0+i*(11*ch)/100,14,(11*ch)/100+.2,fillColor=colors.Color(*rgb),strokeColor=None))
for v in [0,4,8,12,13]:
    yy=y0+v/13*11*ch;d.add(txt(563,yy-3,str(v),fontSize=9))
vertical(d,618,204,'CO2 reduction relative to BAU (Mt)',10)
export_figure('Figure_5_3_environment_grid',d)

tech=['19_eCoal','20_eGas','21_eOil','22_eNUC','23_eREN']
gview=gen.loc[gen.policy_id.isin(POLICIES)].copy()
gview['policy_label']=gview.policy_id.map(NAMES)
gview.to_csv(PLOT_DATA/'Figure_5_4_generation.csv',index=False,encoding='utf-8-sig')
gmat=gview.pivot(index='industry',columns='policy_id',values='change_pct').reindex(index=tech,columns=POLICIES)
d=canvas(680,300)
d.add(txt(60,280,'Generation responses under government-savings-preserving designs',fontName='Helvetica-Bold',fontSize=12.5,fillColor=NAVY))
legend(d,[NAMES[p] for p in POLICIES],PALETTE,y=257,x=73,gap=142)
bars(d,64,61,568,172,[gmat[p].tolist() for p in POLICIES],['Coal','Gas','Oil','Nuclear','REN & other'],PALETTE,(-8,52),[-5,0,10,20,30,40,50])
vertical(d,20,153,'Generation output change relative to BAU (%)',10)
export_figure('Figure_5_4_generation',d)
env_table=s.set_index('policy_id').loc[POLICIES].copy()
env_table['Policy']=[NAMES[x] for x in env_table.index]
env_table[['Policy','HH_ELEC_GWH_change_pct','CO2_reduction_Mt','power_CO2_reduction_Mt','other_CO2_reduction_Mt']].to_csv(OUT/'Table_5_2_environment.csv',index=False,encoding='utf-8-sig')
display(env_table[['Policy','HH_ELEC_GWH_change_pct','CO2_reduction_Mt','power_CO2_reduction_Mt','other_CO2_reduction_Mt']])

## 3 Equity outcomes
환류율 90%에서 REN 몫을 바꾸며 U/I/IC를 비교합니다. REN 100%는 HH가 0이므로 같은 끝점입니다.
저소득=공통 표본의 균등화소득 Q1·Q2, 고제약=constraint_score >= 0.5.
순부담=추가 청구액−현금환류. 음수는 순지출 감소이며 음의 전력가격이 아닙니다.

In [ ]:
li=low.loc[low.rate.eq('r90')&low.low_income].copy()
ji=joint.loc[joint.rate.eq('r90')&joint.low_income&joint.high_constraint].copy()
plot_rows=[]
for group,frame in [('low_income',li),('low_income_high_constraint',ji)]:
    for allocation in ALLOC:
        v=frame.loc[frame.allocation.isin([allocation,'none'])].sort_values('ren_fraction').copy()
        v['plot_allocation']=allocation;v['group']=group
        v['burden_increase_pct']=100*v.burden_increase_share
        plot_rows.append(v)
equity_lines=pd.concat(plot_rows,ignore_index=True)
equity_lines.to_csv(PLOT_DATA/'Figure_5_5_protection.csv',index=False,encoding='utf-8-sig')
d=canvas(680,330)
d.add(txt(66,309,'Coverage of household protection at a 90% recycling share',fontName='Helvetica-Bold',fontSize=13,fillColor=NAVY))
legend(d,ALLOC,[ACOL[a] for a in ALLOC],y=286,x=160,gap=130)
for group,x,title in [('low_income',65,'(a) Low-income households'),('low_income_high_constraint',387,'(b) Low-income / high constraint')]:
    d.add(txt(x,263,title,fontName='Helvetica-Bold',fontSize=10))
    series=[]
    for allocation in ALLOC:
        v=equity_lines.loc[equity_lines.group.eq(group)&equity_lines.plot_allocation.eq(allocation)]
        series.append(list(zip(v.ren_fraction*100,v.burden_increase_pct)))
    lineplot(d,x,62,239,183,series,(0,100),(0,100),[0,20,40,60,80,100],[0,20,40,60,80,100],[ACOL[a] for a in ALLOC],[None,None,[4,2]])
vertical(d,16,160,'Households with higher net bills (%)',10)
d.add(txt(345,21,'REN allocation share a (%)',textAnchor='middle',fontSize=11))
export_figure('Figure_5_5_equity_coverage',d)

q=quintile.loc[quintile.policy_id.eq('r90_a60')&quintile.allocation.isin(ALLOC)].copy()
q['net_change_thousand_won']=q.net_change_won/1000;q['burden_increase_pct']=100*q.burden_increase_share
q.to_csv(PLOT_DATA/'Figure_5_6_quintiles.csv',index=False,encoding='utf-8-sig')
d=canvas(680,330)
d.add(txt(65,309,'Income-group incidence under design C (r=90%, a=60%)',fontName='Helvetica-Bold',fontSize=13,fillColor=NAVY))
legend(d,ALLOC,[ACOL[a] for a in ALLOC],y=286,x=160,gap=130)
for col,x,title,yr,yt in [('net_change_thousand_won',65,'(a) Mean net bill change (KRW thousand)',(-2,12),[-2,0,2,4,6,8,10,12]),
                         ('burden_increase_pct',387,'(b) Households with higher net bills (%)',(0,100),[0,20,40,60,80,100])]:
    d.add(txt(x,263,title,fontName='Helvetica-Bold',fontSize=9.2))
    series=[list(zip(v.income_quintile,v[col])) for allocation in ALLOC for v in [q.loc[q.allocation.eq(allocation)].sort_values('income_quintile')]]
    lineplot(d,x,62,239,183,series,(1,5),yr,[1,2,3,4,5],yt,[ACOL[a] for a in ALLOC],[None,None,[4,2]])
d.add(txt(345,21,'Equivalised-income quintile',textAnchor='middle',fontSize=11))
export_figure('Figure_5_6_income_incidence',d)

pairs=[('TOU','none'),('r90_a00','I'),('r90_a00','IC'),('r90_a60','U'),('r90_a60','I'),('r90_a60','IC'),('r90_a100','none')]
eq_rows=[]
for pid,allocation in pairs:
    l=low.loc[low.policy_id.eq(pid)&low.low_income&low.allocation.eq(allocation)].iloc[0]
    j=joint.loc[joint.policy_id.eq(pid)&joint.low_income&joint.high_constraint&joint.allocation.eq(allocation)].iloc[0]
    eq_rows.append(dict(Policy=NAMES[pid],Allocation=allocation,low_mean_net_won=l.net_change_won,
        low_burden_increase_pct=l.burden_increase_share*100,
        low_high_constraint_mean_net_won=j.net_change_won,
        low_high_constraint_burden_increase_pct=j.burden_increase_share*100))
equity_table=pd.DataFrame(eq_rows)
equity_table.to_csv(OUT/'Table_5_3_equity.csv',index=False,encoding='utf-8-sig')
display(equity_table)

## 4 Economic outcomes
대표가계 EV, GDP, 소비자물가, 실질소비는 서로 다른 지표입니다.
정부저축은 한전 영업이익·누적적자 개선액이 아닙니다.

In [ ]:
econ=s.loc[s.rate.isin(['r70','r80','r90','r100'])].copy()
econ.to_csv(PLOT_DATA/'Figure_5_7_welfare_allocation.csv',index=False,encoding='utf-8-sig')
d=canvas(680,330)
d.add(txt(66,309,'Household welfare across recycling and allocation choices',fontName='Helvetica-Bold',fontSize=13,fillColor=NAVY))
rate_names=['r70','r80','r90','r100'];cols=[colors.HexColor(x) for x in ['#AAB8C5','#668DAA','#173B63','#A85233']]
legend(d,['r=70%','r=80%','r=90%','r=100%'],cols,y=285,x=90,gap=140)
series=[list(zip(v.ren_fraction*100,v.EV_PCT_CTH)) for rate in rate_names for v in [econ.loc[econ.rate.eq(rate)].sort_values('ren_fraction')]]
lineplot(d,69,62,558,195,series,(0,100),(-.15,-.06),[0,20,40,60,80,100],[-.15,-.13,-.11,-.09,-.07],cols,[None,None,None,[4,2]])
vertical(d,18,160,'EV / benchmark consumption (%)',10)
d.add(txt(348,21,'REN allocation share a (%)',textAnchor='middle',fontSize=11))
for label,pid,dx,dy in [('A','r100_a50',-17,12),('C','r90_a60',12,-15)]:
    r=s.loc[s.policy_id.eq(pid)].iloc[0];xx=69+r.ren_fraction*558;yy=62+(r.EV_PCT_CTH+.15)/.09*195
    d.add(Line(xx,yy,xx+dx,yy+dy,strokeColor=GREY,strokeWidth=.7))
    d.add(txt(xx+dx,yy+dy,label,fontName='Helvetica-Bold',fontSize=10))
export_figure('Figure_5_7_economic_welfare',d)

erep=s.set_index('policy_id').loc[POLICIES].copy();erep['Policy']=[NAMES[x] for x in erep.index]
erep[['Policy','GDP_REAL_change_pct','CPI_change_pct','CTH_DEFLATED_change_pct','EV_PCT_CTH','SG_change_billion_USD']].to_csv(OUT/'Table_5_4_economy.csv',index=False,encoding='utf-8-sig')
erep.to_csv(PLOT_DATA/'Figure_5_8_macro_indicators.csv',encoding='utf-8-sig')
d=canvas(680,300)
d.add(txt(63,279,'Macroeconomic adjustments under representative designs',fontName='Helvetica-Bold',fontSize=13,fillColor=NAVY))
for col,x,title,yr,yt in [('GDP_REAL_change_pct',60,'(a) Real GDP (%)',(-.18,0),[-.18,-.12,-.06,0]),
                        ('CPI_change_pct',275,'(b) Consumer prices (%)',(0,.32),[0,.1,.2,.3]),
                        ('CTH_DEFLATED_change_pct',490,'(c) Real consumption (%)',(-.30,0),[-.3,-.2,-.1,0])]:
    d.add(txt(x,249,title,fontName='Helvetica-Bold',fontSize=10))
    bars(d,x,62,162,167,[erep[col].tolist()],[NAMES[p] for p in POLICIES],[NAVY],yr,yt)
d.add(txt(340,20,'Changes relative to BAU',textAnchor='middle',fontSize=10))
export_figure('Figure_5_8_macro_indicators',d)
display(erep[['Policy','GDP_REAL_change_pct','CPI_change_pct','CTH_DEFLATED_change_pct','EV_PCT_CTH','SG_change_billion_USD']])

## 출력
PDF/SVG는 벡터입니다. PNG는 pdftoppm이 있으면 300dpi로 생성합니다.
도표 원자료는 figure_data, 완성 그림·표는 exports에 저장됩니다.

In [ ]:
import shutil
import subprocess
poppler=shutil.which('pdftoppm')
if poppler is None:
    pass  # Install pdftoppm and add it to PATH for optional PNG exports.
if poppler:
    for name in FIGURES:
        subprocess.run([poppler,'-r','300','-png','-singlefile',str(OUT/(name+'.pdf')),str(OUT/name)],check=True,capture_output=True)
else:
    print('PDF/SVG 생성 완료. PNG는 pdftoppm 설치 후 생성할 수 있습니다.')
print('결과 폴더:',OUT)